In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from dotenv import load_dotenv
import os 
load_dotenv()

openai_client = OpenAI()

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant_openai = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
    model='gpt-5.4-mini',
    api_type='openai'
)

assistant_groq = RAGBase(
    index=index,
    llm_client=groq_client,
    instructions=instructions,
    model='qwen/qwen3.6-27b',
    api_type='groq'
)

In [4]:
answer = assistant_openai.rag('How do I run Ollama locally?')
print("OpenAI Answer:")
print(answer)

answer = assistant_groq.rag('How do I run Ollama locally?')
print("Groq Answer:")
print(answer)



OpenAI Answer:
To run Ollama locally:

1. Install Ollama from: https://ollama.com/download  
   - macOS: download the `.pkg`
   - Windows: download the `.msi`
   - Linux: run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. Open a terminal and start a model:
```bash
ollama run llama3
```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

3. To test that the local server is running, use:
```bash
curl http://localhost:11434
```

You should get a response like:
```json
{"models": [...]}  
```

If you get a connection refused error while prompting Ollama in the homework, restart the server with:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```
Groq Answer:

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** How do I run Ollama locally?
   - **Context:** Provided FAQ snippets about Ollama installation, running it, testing it, Python client usage, connection refused error recovery, and its compati

In [14]:
messages = [
    {"role": "user", "content": "What was my last question ?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

print(response.output_text)

Your last question was: **“What was my last question ?”**


In [15]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [16]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for answers to user questions",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [17]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output)

[ResponseOutputMessage(id='msg_0f12ea35e1cb9204006a8f1801f67887d2b8db4790561e9820', content=[ResponseOutputText(annotations=[], text='Your last question was: **“What was my last question ?”**', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]


In [19]:
import json
call = response.output[0]
args = json.loads(call.arguments)
print(args)

results = search(**args)
result_json = json.dumps(results, indent=2)
print(result_json)

AttributeError: 'ResponseOutputMessage' object has no attribute 'arguments'

In [11]:
messages.extend(response.output)

messages.append({
    "type" : "function_call_output",
    "call_id" : call.call_id,
    "output" : result_json
})


In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes — you can still join the course anytime and start learning.

If you want a certificate, you’ll need to submit your project while submissions are still open.
